# Chapter 3 — Dictionaries and Sets
### Interactive Teaching Notebook

One topic at a time. Predict before you run.

## Topic 1 — Mordern dict Syntax 

- Modern dict syntax refers to the modern, concise ways Python provides to create, unpack, and merge dictionaries without resorting to verbose loops or multiple boilerplate method calls. This includes **dict comprehensions (dictcomps)**, **mapping unpacking** using the double-star operator (`**`), and the **dictionary union operators** (`|` and `|=`) introduced in Python 3.9.

- A **dictcomp** builds a `dict` by taking `key: value` pairs from any iterable — the same idea as a listcomp, but the result is a mapping instead of a sequence. The general form is:`{key_expr: value_expr for item in iterable}`

**Why dictcomps exist (the mechanism):** you can pass an iterable of `(key, value)` pairs directly to `dict()`, but you cannot transform them — `dict()` takes the pairs as-is. A dictcomp lets you use any expression for the key and any expression for the value, so you can swap, filter, or compute on the fly during construction. For example, `{country: code for code, country in data}` turns a `(code, country)` iterable into a `{country: code}` dict — something `dict(data)` cannot do directly.

**Gotcha:** duplicate keys are silently resolved — the last occurrence wins. If two items produce the same key, the earlier value is overwritten without any error.

In [3]:
# Run this after you've answered the prediction question below.
scores = [('alice', 88), ('bob', 74), ('carol', 95), ('alice', 92)]
all_socres = dict(scores)
print(all_socres)

# Build a dict mapping name -> score, but only for scores >= 80
high_scores = {name: score for name, score in scores if score >= 80}

# Now build the *inverse*: score -> name, for the same filtered set
inverse = {score: name for name, score in scores if score >= 80}

print(high_scores)
print(inverse)

{'alice': 92, 'bob': 74, 'carol': 95}
{'alice': 92, 'carol': 95}
{88: 'alice', 95: 'carol', 92: 'alice'}


## Topic 1.2 — Unpacking Mappings

The ** operator unpacks key-value pairs from an existing mapping into a new dictionary literal. When duplicate keys exist during literal construction, the rightmost value overwrites any previous values. The chapter covers two places you can use it:

**1. In a function call** — you can unpack multiple mappings with `**`, but all keys across all arguments must be strings and must be unique (duplicate keyword arguments are a syntax error) because Python doesn't allow duplicate keyword argument a function and it can't bind non-string argument as a keyword argument.

```python
dump(**{'x': 1}, y=2, **{'z': 3})   # fine — all keys unique strings
dump(**{'x': 1}, **{'x': 2})        # TypeError — duplicate key 'x'
```

**2. Inside a `dict` literal** — also allowed multiple times. Here, **duplicate keys are allowed**; the later occurrence simply overwrites the earlier one:

```python
{'a': 0, **{'x': 1}, 'y': 2, **{'z': 3, 'x': 4}}
# 'x' appears twice — the second value (4) wins
```

This is also a way to merge mappings (though `|` — topic 3 — is usually cleaner for that).

In [38]:
# Run after answering the prediction question below.
def dump(**kwargs):
    return kwargs

a = {'flavor': 'vanilla', 'size': 'large'}
b = {'size': 'small', 'topping': 'sprinkles'}

# Unpack both into a dict literal; 'size' appears in both
merged = {**a, **b}

print(dump(**a, topping='nuts'))   # function call — unique keys only
print(merged)                      # dict literal — duplicate key 'size'

{'flavor': 'vanilla', 'size': 'large', 'topping': 'nuts'}
{'flavor': 'vanilla', 'size': 'small', 'topping': 'sprinkles'}


In [ ]:
a = {'x':1, 'y': 2}
b = {'x': 3}
c = {**a, **b} # later keys win!
c

{'x': 3, 'y': 2}

In [ ]:
# unpacking in a function call with keys not being string raise TypeError: keywords must be strings
c = {1: 'a', 2: 'b'}
dump(**c)

TypeError: keywords must be strings

## Topic 1.3 — Union Operators (| and |=)

- Python 3.9 added two operators for merging dicts, borrowing from the **set union** notation. 
    - The | operator performs a merge, creating a brand-new dictionary of the same type as the left operand. Under the hood, this delegates to the __or__ special method. 
    - The |= operator performs an in-place update (modifying the left-hand mapping directly) and maps to the __ior__ in-place special method



| Operator | What it does |
|---|---|
| `d1 \| d2` | Creates a **new** dict — `d1` is unchanged |
| `d1 \|= d2` | Updates `d1` **in place** |

In both cases, when a key exists in both dicts, the **right-hand operand wins** — same rule as `{**d1, **d2}`.

One subtlety from the chapter: the type of the result from `d1 | d2` is usually the type of the **left** operand. (This matters if you subclass `dict`, covered later in Chapter 16.)

In [4]:
# Run after answering the prediction question below.
defaults = {'theme': 'light', 'font': 'serif', 'size': 12}
overrides = {'font': 'sans', 'size': 14}

merged = defaults | overrides      # new dict
defaults |= overrides              # in-place update

print(merged)
print(defaults)
print(merged is defaults)          # same object?

{'theme': 'light', 'font': 'sans', 'size': 14}
{'theme': 'light', 'font': 'sans', 'size': 14}
False


In [4]:
d1 = {1: "apple", "type": "fruit"}
d2 = {1.0: "banana", "organic": True}

#NOTE: key of 1 and 1.0 are treated as they're the same and merged into a key of 1 (int).
merged = d1 | d2 
merged, type(list(merged.keys())[0])

({1: 'banana', 'type': 'fruit', 'organic': True}, int)

## Gotchas & Edge Cases (The Non-Obvious)
- Function Unpacking Key Restrictions: You can unpack keys of any hashable type (like integers or tuples) when merging dicts inside a literal—e.g., {**d1, **d2}. However, if you attempt to use the exact same unpacking operator to pass keyword arguments to a function—e.g., func(**d1)—Python strictly requires that all keys in the unpacked dictionary are strings and are unique. If you pass non-string keys, Python raises a TypeError.
- Key Preservation vs. Value Update: When merging two dictionaries that contain keys that compare equal (e.g., the integer 1 and the float 1.0), Python recognizes them as duplicate entries because they share the same hash and are equal. During the merge, the value is updated to the rightmost operand's value, but the original key object from the left-hand dictionary is preserved. This is a CPython memory optimization to avoid shifting hash table buckets unnecessarily.


## Topic 2 — Pattern Matching with Mappings

`match/case` can match mapping subjects. A few critical rules from the chapter:

**1. Partial matches succeed.**
A case like `{'type': 'book'}` will match even if the subject has extra keys like `'title'` or `'pages'`. This is the opposite of sequence patterns, which require the length to match.

**2. Key order in the pattern is irrelevant.**
`{'api': 2, 'type': 'book'}` matches the same subjects as `{'type': 'book', 'api': 2}`.

**3. Capture remaining keys with `**rest`.**
You can add one `**variable` at the end to catch all unmatched keys as a dict. `**_` is explicitly forbidden (it would be redundant).

**4. `__missing__` is NOT triggered.**
Pattern matching uses `d.get(key, sentinel)` internally — not `d[key]` — so `defaultdict` auto-creation never fires during a match.

**5. Patterns match any `collections.abc.Mapping` subclass**, not just `dict` — so `OrderedDict` works too.

In [ ]:
# Run after answering the prediction question below.
def classify(record):
    match record:
        case {'type': 'order', 'item': item, **rest}:
            return f"order for {item!r}, extra={rest}"
        case {'type': 'order'}:
            return "order with no item"
        case _:
            return "unknown"

r1 = {'type': 'order', 'item': 'book', 'qty': 3, 'urgent': True}
r2 = {'type': 'order', 'status': 'pending'} # partial matching!
r3 = {'type': 'payment', 'amount': 99}

print(classify(r1))
print(classify(r2))
print(classify(r3))



def classify2(record):
    match record: 
        case {'type': 'order'}: # order matters here!
            return "order with no item"
        case {'type': 'order', 'item': item, **rest}:
            return f"order for {item!r}, extra={rest}"
        case _:
            return "unknown"

r1 = {'type': 'order', 'item': 'book', 'qty': 3, 'urgent': True}
r2 = {'type': 'order', 'status': 'pending'}

print(classify2(r1))
print(classify2(r2))

order for 'book', extra={'qty': 3, 'urgent': True}
order with no item
unknown
order with no item
order with no item


In [5]:
from collections import defaultdict

# defaultdict auto-creates a value when you access a missing key via d[key]
dd = defaultdict(list)

# d[key] would create the key — but match/case uses d.get(key, sentinel) instead
match dd:
    case {'color': c}:
        print(f"matched, color={c}")
    case _:
        print("no match")

# Prove the key was NOT auto-created during the match
print("'color' in dd:", 'color' in dd)  # False — match never triggered __missing__


data = defaultdict(lambda: "fallback")
data['type'] = 'book'
data['title'] = 'Fluent Python'

# Evaluate matching pattern
match data:
    case {'type': 'book', 'author': author_val}:
        result = f"Author: {author_val}"
    case {'type': 'book'}:
        result = f"Book title: {data['title']}"
    case _:
        result = "No match"

# 1. What will this print?
print(result)

# 2. What will this print?
print('author' in data)

no match
'color' in dd: False
Book title: Fluent Python
False


Suppose you have a dictionary d and you want to dynamically check for a key whose name is stored in a variable, target_key.

**Keys** in mapping patterns must be literals or dotted names (representing constants); they cannot be simple variable names.

Why does Python restrict this?

1. The Ambiguity of Capture Patterns: In Python's pattern matching, simple names (like val or target_key) are treated as capture patterns—they bind whatever value is matched to that name.

2. Hash Lookup Requirement: To match a mapping, Python must be able to perform a hash lookup on the subject dictionary before binding any pattern variables. If a key in a pattern were a simple variable name, it would be ambiguous: is Python supposed to look up the key 'target_key' literal, look up the value of the variable target_key defined outside, or bind the matched key to a new local variable named target_key?

3. The Workaround: If you must match keys against constants defined elsewhere, those constants must be dotted names (e.g., constants.TARGET_KEY). When Python encounters a dot in a pattern key, it looks up the value of that attribute at runtime instead of treating it as a variable binding.

In [49]:
target_key = 'a'
d = {'a': 1, 'b': 2}

match d:
    case {target_key: val}:
        print(f"Found value: {val}")
    case _:
        print("No match")

SyntaxError: invalid syntax (1234222505.py, line 5)

## Topic 3 — Standard API of Mapping Types & Hashability

- The Mapping API refers to the collection of standard methods and interfaces (defined in collections.abc.Mapping and MutableMapping) that all Python dictionaries and custom mappings implement. Hashability is the strict standard that determines which Python objects are allowed to act as keys within these mappings

- Python's dictionaries are powered by hash tables, a high-performance data structure that achieves $O(1)$ lookup. 
    1. When you set d[key] = value, Python calls hash(key) to get a hash integer. It uses this integer to find a specific bucket in its memory array. 
    
    1. If you modify a key after it has been added to the dictionary, its hash value would change. If you later tried to look up the key, its new hash would point to a different bucket, making the original entry permanently lost inside the dictionary. 
    
    1. To prevent this corruption, Python requires keys to be hashable. An object is hashable if: 
        - It implements __hash__() and __eq__(). 
        - Its hash value remains unchanged throughout its entire lifetime. 
        - If two objects compare equal (a == b), they must have the exact same hash value (hash(a) == hash(b)).
        
        To optimize inserting or updating keys mapped to mutable values (like appending to a list inside a dictionary), the API offers setdefault(). Instead of running separate containment checks, retrievals, and insertions, d.setdefault(key, default) performs the entire operation in a single, optimized hash table lookup

- Gotchas & Edge Cases (The Non-Obvious)
    
    Default User-Defined Hashability: By default, all user-defined class instances are hashable. Their hash value is derived from their memory address (id()), and they compare equal only to themselves. However, the moment you override __eq__ to compare instance values, Python automatically disables default hashability (setting __hash__ to None) to preserve the contract that equal objects must have equal hashes.

    The "Relative" Tuple Hashability: Tuples are immutable, but they are not guaranteed to be hashable. A tuple is only hashable if every single nested element inside it is also hashable. For instance, (1, 2, [16, 17]) is unhashable because it contains a mutable list.

    **The setdefault Performance Cost**: While setdefault optimizes lookup steps, its default argument (the second parameter) is **eagerly** evaluated on every single execution. If your fallback value involves a database query or an expensive instantiation (e.g., `d.setdefault(key, fetch_heavy_default())`), you will pay that performance cost every time the line runs, even if the key already exists

The `collections.abc` module defines two ABCs that describe the mapping interface:

- **`Mapping`** — read-only interface (`__getitem__`, `__len__`, `__iter__`, plus derived methods like `get`, `keys`, `values`, `items`)
- **`MutableMapping`** — adds write operations (`__setitem__`, `__delitem__`, plus `pop`, `update`, `setdefault`, etc.)

**Why `isinstance(x, abc.Mapping)` over `isinstance(x, dict)`:**
Checking for `dict` rejects any custom mapping that doesn't inherit from `dict`. Checking `abc.Mapping` accepts any object that implements the mapping interface, regardless of its inheritance chain.

**Building custom mappings — why `UserDict` over `dict` (the mechanism):**
`dict` is a C built-in with implementation shortcuts: some of its methods call each other internally in C, bypassing any Python override you write. For example, `dict.update()` may internally skip your custom `__setitem__`. The chapter states: *"the built-in has some implementation shortcuts that end up forcing us to override methods that we can just inherit from UserDict with no problems."* `UserDict` is pure Python and holds all items in an internal dict called `.data`, routing every operation through your overrides predictably.

**Gotcha:** subclassing `dict` directly can silently ignore your overrides — the bug only appears in specific call paths, making it hard to diagnose.

In [12]:
from collections import abc

# A minimal custom mapping that subclasses abc.Mapping — NOT dict
class ReadOnlyMap(abc.Mapping):
    def __init__(self, data):
        self._data = data
    def __getitem__(self, key):
        return self._data[key]
    def __iter__(self):
        return iter(self._data)
    def __len__(self):
        return len(self._data)

m = ReadOnlyMap({'x': 1, 'y': 2})

print(isinstance(m, dict))           # False — not a dict subclass
print(isinstance(m, abc.Mapping))    # True  — satisfies the Mapping interface

# A function checking dict would wrongly reject m:
def bad_lookup(d, key):
    if not isinstance(d, dict):
        raise TypeError("expected a dict")
    return d[key]

def good_lookup(d, key):
    if not isinstance(d, abc.Mapping):
        raise TypeError("expected a mapping")
    return d[key]

print(good_lookup(m, 'x'))   # works fine
try:
    bad_lookup(m, 'x')       # raises TypeError even though m behaves like a dict
except TypeError as e:
    print(f"bad_lookup failed: {e}")

False
True
1
bad_lookup failed: expected a dict


## Topic 3.2 — What Is Hashable

From the chapter's definition: an object is **hashable** if:
1. It has a `__hash__()` method that returns a hash code that **never changes** during its lifetime, AND
2. It has an `__eq__()` method to compare with other objects.

Additionally: **hashable objects that compare equal must have the same hash code.**

**Rules for built-in types:**
- Numeric types, `str`, `bytes` — always hashable (flat and immutable)
- `frozenset` — always hashable (every element it contains must be hashable by definition)
- `tuple` — hashable **only if all its items are hashable**
- `list`, `set`, `dict` — never hashable (mutable)

**User-defined types** are hashable by default: their hash code is `id(self)`, and the inherited `__eq__` compares object identities. The moment you define a custom `__eq__` that looks at internal state, you must also define a consistent `__hash__` — otherwise Python sets `__hash__` to `None` and the object becomes unhashable.

**Note:** hash codes may differ across Python processes due to a security salt — they are only guaranteed constant within one process.

In [13]:
# Run after answering the prediction question below.
candidates = [
    42,
    'hello',
    (1, 2, 3),
    (1, 2, [3, 4]),        # tuple containing a list
    frozenset([1, 2, 3]),
    frozenset([1, frozenset([2, 3])]),
]

for obj in candidates:
    try:
        print(f"{repr(obj):35} -> hash={hash(obj)}")
    except TypeError as e:
        print(f"{repr(obj):35} -> UNHASHABLE: {e}")

42                                  -> hash=42
'hello'                             -> hash=-4498581224842078346
(1, 2, 3)                           -> hash=529344067295497451
(1, 2, [3, 4])                      -> UNHASHABLE: unhashable type: 'list'
frozenset({1, 2, 3})                -> hash=-272375401224217160
frozenset({1, frozenset({2, 3})})   -> hash=2631659458078453944


In [8]:
class Token:
    def __init__(self, val):
        self.val = val

t1 = Token("apple")
t2 = Token("apple")

# We construct a standard dictionary
registry = {}

# We populate the registry
registry[t1] = "first"
registry[t2] = "second"

# 1. What will this print?
print(len(registry))

# Now we try to construct a compound key
try:
    combo = (t1, [20, 21])
    registry[combo] = "combo"
    print("Success")
except TypeError as e:
    # 2. What exception name is caught here?
    print(e)

2
unhashable type: 'list'


In [ ]:
class MutableType:
    def __init__(self, x):
        self.x = x

    def update(self, v) -> None:
        self.x = v
        return None

class MutableType2:
    def __init__(self, x):
        self.x = x

    def update(self, v) -> None:
        self.x = v
        return None

    #NOTE Unhashable because eq is implemented but hash isn't!
    def __eq__(self, value):
        return self.x == value.x

mt = MutableType(1)
mt2 = MutableType2(2)
try:
    d = {mt:1, mt2:2}
except TypeError as e:
    print(e)

print(d[1].x)
mt.update(2)
mt2.update(3)
d[1].x, d[2].x

unhashable type: 'MutableType2'
2


(2, 3)

In [34]:
# mt2 was used as a dict VALUE — values are never hashed, so no error
# Test hashability directly:
print(hash(mt))    # MutableType has no custom __eq__ → still hashable

try:
    print(hash(mt2))   # MutableType2 has custom __eq__ → should be unhashable
except TypeError as e:
    print(f"TypeError: {e}")

# Also try using mt2 as a dict KEY:
try:
    test = {mt2: 'hello'}
except TypeError as e:
    print(f"As key: {e}")

278840057
TypeError: unhashable type: 'MutableType2'
As key: unhashable type: 'MutableType2'


## Topic 7 — Inserting or Updating Mutable Values (`setdefault`)

When building a dict whose values are mutable (e.g. lists), you often need to: get the existing list for a key, or create a new one if the key is missing, then append to it.

**The naive `if/in` approach** (2 lookups when key exists, 3 when missing):
```python
if word not in index:           # lookup 1
    index[word] = []            # lookup 2 — only when key is missing
index[word].append(location)   # lookup 2 or 3 — always
```

**The `setdefault` approach** (1 lookup always):
```python
index.setdefault(word, []).append(location)
```

**The mechanism:** `setdefault(key, default)` does two things atomically in one lookup:
- If `key` is already in the dict → return `d[key]` (default is ignored)
- If `key` is **not** in the dict → insert `d[key] = default` **and return it**

Because it returns the value, you can chain `.append()` directly on the result — no second lookup needed.

**Gotcha:** `d.get(key, [])` looks similar but is *not* equivalent. It returns the fallback `[]` without inserting it into the dict, so you still need `d[key] = ...` afterwards — two lookups. `setdefault` inserts and returns in one shot.

In [15]:
# Run after answering the prediction question below.
index = {}

words = [('the', 1), ('cat', 1), ('the', 2), ('cat', 3), ('sat', 2)]

for word, line in words:
    index.setdefault(word, []).append(line)

print(index)

# Now check: what does setdefault return when the key already exists?
existing = index.setdefault('cat', ['SHOULD_NOT_APPEAR'])
print(existing)         # returns the existing list — default is ignored
print(index['cat'])     # dict is unchanged

{'the': [1, 2], 'cat': [1, 3], 'sat': [2]}
[1, 3]
[1, 3]


In [20]:
index.get('cat', []), index.get('foo', []), 'foo' in index

([1, 3], [], False)

## Topic 8 — `defaultdict` and `__missing__`

### `defaultdict`

`collections.defaultdict` creates items with a default value on demand whenever a missing key is looked up via `d[key]`. You provide a **callable** at construction time — called `default_factory` — which is called with no arguments to produce the default value.

```python
dd = defaultdict(list)
dd['new-key'].append(1)   # auto-creates dd['new-key'] = [] first, then appends
```

**The mechanism:** `defaultdict` works by implementing `__missing__`. When `d[key]` triggers `__getitem__` and the key isn't found, Python calls `__missing__(key)`. In `defaultdict`, that method calls `default_factory()`, inserts the result under `key`, and returns it.

**Critical gotcha — `default_factory` is only triggered by `d[key]`, not by `d.get(key)`:**
- `dd['x']` → triggers `__missing__`, inserts and returns the default
- `dd.get('x')` → returns `None` (or your fallback), does NOT insert, does NOT call `__missing__`
- `'x' in dd` → returns `False`, does NOT insert

### `__missing__` directly

`__missing__` is not defined on `dict`, but `dict.__getitem__` knows to call it if it exists on a subclass. This lets you customise missing-key behaviour without `defaultdict`. The chapter's example converts non-string keys to `str` on lookup:

```python
class StrKeyDict0(dict):
    def __missing__(self, key):
        if isinstance(key, str):   # already a str and still missing → real KeyError
            raise KeyError(key)
        return self[str(key)]      # try again with str version
```

**Gotcha — the `isinstance` guard is essential.** Without it, `self[str(key)]` would call `__getitem__` again with a `str` key → trigger `__missing__` again → infinite recursion.

In [ ]:
# Run after answering the prediction question below.
from collections import defaultdict

dd = defaultdict(list)

dd['a'].append(1)
dd['a'].append(2)
dd['b'].append(9)

print(dd['a'])          # existing key -> [1, 2]
print(dd['c'])          # missing key — what happens? -> []
print(dd.get('d'))      # missing key via .get() — what happens? -> None!
print(list(dd.keys()))  # which keys exist now? -> ['a', 'b', 'c']

[1, 2]
[]
None
['a', 'b', 'c']


In [9]:
class MissingDict(dict):
    def __missing__(self, key):
        if isinstance(key, str):
            raise KeyError(key)
        return self[str(key)]

d = MissingDict()
d["code"] = 404

# 1. What will this print?
print(d.get(10, "fallback"))

# 2. What will this print?
print(d[8])

fallback


KeyError: '8'

In [ ]:
from collections import UserDict

class MissingUserDict(UserDict):
    def __missing__(self, key):
        if isinstance(key, str):
            raise KeyError(key)
        return self[str(key)]

d = MissingUserDict()
d["code"] = 404
print(d.get(10, "fallback")) # UserDict overrides .get() and does NOT trigger __missing__!

fallback


## Topic 5 Variations of dict
Standard Python dictionaries are incredibly powerful, but the standard library's collections module provides several specialized variations of dict designed for specific architectural tasks: `OrderedDict`, `ChainMap`, `Counter`, and `UserDict`.

The Mechanism: How They Work
- `collections.OrderedDict`: Prior to Python 3.6, standard dictionaries did not preserve key insertion order. `OrderedDict` was created to maintain this order. While standard dict now preserves insertion order by default, `OrderedDict` remains useful because it is specifically optimized for frequent reordering operations (offering methods like move_to_end() and a predictable double-ended queue layout under the hood).
- `collections.ChainMap`: A ChainMap groups multiple dictionaries or mappings together into a single, searchable interface. When you look up a key, Python searches each dictionary in the chain sequentially from **left to right**, returning the value from the first dictionary where the key is found.
- `collections.Counter`: A mapping designed for counting hashable objects. Every key maps to an integer count. It overrides standard dictionary behavior to support multiset arithmetic (such as using + and - to combine or subtract tallies)

Gotchas & Edge Cases (The Non-Obvious)
- The ChainMap Mutation Asymmetry: While lookup operations scan through the entire chain of dictionaries, write mutations (insertions, updates, and deletions) only ever target the **very first dictionary in the chain**. For example, if you look up a configuration setting that exists in a default dictionary down the chain and attempt to delete or update it, Python will mutate the first dictionary instead, leaving the lower dictionary completely untouched.

- OrderedDict Equality Trap: In standard Python, two dict objects with the same keys and values are equal (d1 == d2 evaluates to True) regardless of their key insertion order. However, OrderedDict takes key order into account during equality comparisons. If the keys were inserted in a different sequence, two otherwise identical OrderedDict objects are not equal.


In [ ]:
# Topic 5 Challenge: Predict the Output
from collections import ChainMap

defaults = {'theme': 'dark', 'font': 'Consolas'}
user_settings = {'theme': 'light'}

settings = ChainMap(user_settings, defaults) #NOTE：user_settings is the 1st map and defaults is the 2nd!

# Step 1: Assign to a key that exists in 'defaults'
settings['font'] = 'Monospace' #NOTE: this actually insert 'font' to user_settings and doesn't update settings!

# Step 2: Delete that same key from settings
del settings['font'] # This deletes font from user_settings as it's the first map in the chain.

# 1. What will this print?
print(settings['font']) # this returns the font from 'defaults' which is still left in the chain.

# 2. What will this print?
print('font' in user_settings) # 'font' doesn't exist in user_settings anymore as it was deleted!

Consolas
False


In [ ]:
from collections import ChainMap

global_scope = {'x': 1, 'y': 2}
local_scope = {}

context = ChainMap(local_scope, global_scope)

# A variable assignment occurs inside the local block:
context['x'] = 10 # local_scope now is {'x': 10}

# Now we spawn a nested block, creating a new inner scope:
nested_scope = {}
nested_context = context.new_child(nested_scope)

# What is the lookup value of 'x' in the nested context?
print(nested_context['x']) # 10 from local_scope

# delete 'x' from the nested context:
del nested_context['x'] # key error as nested scope has no key of x.




10


KeyError: "Key not found in the first mapping: 'x'"

## Topic 6: Immutable Mappings

- What It Is

In the Python standard library, all built-in mapping types (like dict, defaultdict, etc.) are mutable. However, there are scenarios (such as exposing a configuration dictionary or an API interface) where you want to allow clients to read a dictionary but prevent them from accidentally modifying it. Python addresses this by offering a read-only dynamic wrapper class called `types.MappingProxyType`

- The Mechanism

Instead of copying the dict or freezing its elements, MappingProxyType builds a read-only dynamic proxy around an existing dictionary. The proxy does not hold its own copy of the keys and values. It holds a reference to the original mutable dictionary.
When a user queries keys through the proxy, it delegates the lookup directly to the underlying dictionary.
Because it is a dynamic proxy, any updates made to the original dictionary are instantly visible through the proxy.
However, any attempt to write, update, or delete a key directly through the proxy (e.g., proxy[key] = value or del proxy[key]) is intercepted and raises a TypeError

- Gotchas & Edge Cases (The Non-Obvious)

Dynamic Views vs. Frozen Mappings: A mapping proxy is not a frozen dict. It is only read-only from the perspective of the proxy holder. If you retain a reference to the underlying dict and update it, the proxy reflects those updates immediately.
Shallow Immutability: Like tuples, a mapping proxy only protects the references it holds. If your dictionary maps to mutable objects (like lists or other dictionaries), you cannot rebind keys on the proxy itself, but you can still mutate the mutable objects retrieved through the proxy!

In [16]:
# Topic 6 Challenge: Predict the Output
from types import MappingProxyType

# We start with a standard dictionary containing a list
data = {'a': 1, 'b': [7, 8]}
proxy = MappingProxyType(data)

# Let's perform some actions:

# Action 1: Mutate the underlying dict directly
data['a'] = 100

# Action 2: Mutate a mutable value through the proxy
try:
    proxy['b'].append(30)
    print("Success A") # this is printed!
except TypeError:
    print("Fail A") 

# Action 3: Rebind a key directly on the proxy
try:
    proxy['a'] = 99
    print("Success B")
except TypeError:
    print("Fail B") # this is printed

Success A
Fail B


In [ ]:
from collections import abc


def load_config(config):
    if not isinstance(config, abc.Mapping): #NOTE can't use dict here to check the type!
        raise TypeError("Config must be a concrete dict instance!")
    return config.get('port', 8080)

from types import MappingProxyType

raw_config = {'port': 9000}
safe_config = MappingProxyType(raw_config)
load_config(safe_config)

9000

## Topic 7: Dictionary Views
1. What It Is

In Python 3, the dictionary methods .keys(), .values(), and .items() do not return static lists or separate iterators. Instead, they return specialized objects called dictionary views (dict_keys, dict_values, and dict_items). These views represent high-performance, memory-efficient, read-only projections of the dictionary's underlying C-level hash table structures.

2. The Mechanism

- Dynamic Proxies: A dictionary view holds a direct reference to the parent dictionary, behaving as a dynamic proxy. It does not duplicate any key or value data in memory. If the source dictionary is mutated (even after the view is created), the changes are instantly visible when reading through the view.
- Set-like Behavior: Because dictionary keys are guaranteed to be unique and hashable by definition, the **dict_keys** and **dict_items** views implement the Python Set abstract base class interface. This allows them to support rich mathematical set operations—such as union (|), intersection (&), difference (-), and symmetric difference (^)—directly with other views or standard Python sets, without manual conversion.

3. Gotchas & Edge Cases (The Non-Obvious)

- The `dict_items` Hashability Trap: While a `dict_keys` view can always be treated as a set (since keys are hashable by contract), a `dict_items` view behaves like a set only if all values currently stored in the dictionary are also hashable. If the dictionary contains even a single unhashable value (like a list or another dict), attempting a set operation on dict_items will raise a TypeError at runtime.
- No dict_values Set Operations: Because dictionary values do not have to be unique or hashable, `dict_values` views do not implement any set operations.
- Return Types of Set Operations: Performing a set operation (like & or |) on a dictionary view does not return another dynamic view. It eagerly evaluates the operation and returns a standard, mutable Python `set` object in memory.

In [24]:
d1 = {'a': 1, 'b': [11, 12]}
d2 = {'b': [11, 12], 'c': 3}

# Operation 1: Intersect keys views
common_keys = d1.keys() & d2.keys()

# Operation 2: Intersect items views
try:
    common_items = d1.items() & d2.items()
    result_status = "Success"
except TypeError:
    result_status = "TypeError"

# 1. What will this print?
print(common_keys) # {'b'}

# 2. What will this print?
print(result_status) # TypeError

{'b'}
TypeError


## Topic 8: Set Theory, Literals, and Set Operations

1. What It Is

    Python's set and its immutable companion frozenset are built-in implementations of mathematical sets. They represent collections of unique, hashable objects and provide native support for standard set-theory operations like union, intersection, difference, and symmetric difference.

1. The Mechanism: Set Representation and Assembly

    - Under the hood, sets are powered by hash tables, very much like dictionaries but containing only keys without associated values.
    - The Compilation Optimization: Defining a set using literal syntax (e.g., `{1, 2, 3}`) is **significantly faster** than using the constructor (e.g., `set()`). When Python parses a literal, the compiler directly executes a specialized `BUILD_SET` bytecode instruction. Conversely, calling `set(...)` requires a slow global name lookup for the set callable, the instantiation of an intermediate list object on the stack, and finally passing that list to the constructor.
    - Resizing and Reordering: Sets do not maintain a stable, reliable element order. When elements are added, if the hash table becomes more than **two-thirds** full, Python resizes the table. During this resizing, all elements are re-hashed and re-inserted, which can dynamically shuffle the apparent order of elements.

1. Gotchas & Edge Cases (The Non-Obvious)
    - The Syntax Quirk: There is no literal notation for an empty set. Writing `{}` creates an empty dict. To instantiate an empty set, you must explicitly call `set()`. Similarly, `frozenset` has no literal notation and must always be built via the `frozenset()` constructor.
    - Operators vs. Methods: Python's set **infix operators** (such as &, |, -, and ^) strictly require both operands to be actual set or frozenset instances. On the other hand, their corresponding named methods (such as .intersection(), .union(), .difference(), etc.) are much more liberal and will happily accept any **iterable** as an argument

In [ ]:
# Imagine you are optimizing a performance-critical data pipeline. You need to verify whether any "blacklisted" IDs exist inside a set of active user IDs. You have two alternatives to choose from:

# Alternative A:
active_ids = {1001, 1002, 1003}
blacklist = {1002, 9999}

# Check intersection
has_blacklist = len(active_ids & blacklist) > 0


# Alternative B:
active_ids = {1001, 1002, 1003}
blacklist = [8]  # Notice this is a list

# Check intersection
has_blacklist = not active_ids.isdisjoint(blacklist)

# B is significantly faster because it doesn't need to allocate any new memory; just looping a list and lookup whether those elements
# exist in a set e.g. highly optimized O(1) hash table lookup! 
# Also it's short circuited e.g. as soon as it finds an inserction element then it will return.
# Whereas A has to create the intersection set fully before it can determine the result.

## Topic 9: The Practical Consequences of How dicts & sets Work

1. What It Is

Standard dictionaries and sets are highly optimized C structures under the hood, but their performance characteristics—and some surprising runtime behaviors—are directly dictated by the mechanics of hash tables. Understanding these details allows you to write memory-efficient code and avoid silent data corruption.

2. The Mechanism: Behind the Table

- Compacted Layout (since 3.6): Modern Python dictionaries use a split-array structure. Instead of maintaining a sparse array of large, empty 24-byte bucket structures to avoid collisions, CPython keeps an extremely dense array containing only the actual keys, hashes, and values in the order they were inserted. A separate, tiny, sparse array of small integers (indices) maps hash buckets to positions in the dense array. This preserves insertion order as a side effect and dramatically reduces memory consumption.
- Key-Sharing Dictionary (since 3.3 / PEP 412): To optimize memory for object-oriented programs, instances of user-defined classes share a single, common hash table stored on the class itself. The individual instance's `__dict__` is kept as a simple, flat array of pointers containing only the attribute values.

3. Gotchas & Edge Cases (The Non-Obvious)

- The "Late Attribute" Memory Penalty: The key-sharing optimization only works if attributes are defined within the __init__ constructor method. If you dynamically assign a new instance attribute to an object after initialization has finished, CPython is forced to instantly discard the shared layout for that object. It de-allocates the shared array and creates a full, heavy, private dictionary in memory just for that single instance, increasing memory usage by 10% to 20% across your program.
- Set Mutation is a Silent Killer: While sets prevent you from inserting unhashable elements (like lists) at runtime, they cannot prevent you from mutating a custom hashable object after it has been added to the set. If you mutate an object's attribute that is used to compute its hash, the object's hash value changes, but its position in the set's internal table remains the same. The object becomes a "ghost"—permanently lost inside the set.

In [25]:
class MutableKey:
    def __init__(self, val):
        self.val = val
        
    def __hash__(self):
        return hash(self.val)
        
    def __eq__(self, other):
        return isinstance(other, MutableKey) and self.val == other.val

k = MutableKey(1)
my_set = {k}

# Statement 1
print(1, k in my_set)

# We modify the key's internal state
k.val = 2

# Statement 2
print(2, k in my_set)

# Statement 3
print(3, any(x is k for x in my_set))


1 True
2 False
3 True


1. The Dynamic Duck-Typing of dict.update()
The way d.update(m) processes its input is a classic masterclass in Pythonic duck typing.
The Mechanism: It does not do a rigid type check. Instead, it first checks if the argument m has a .keys() method. If it does, update() treats it as a mapping. If it does not, the method falls back to iterating over m, assuming that it yields (key, value) pairs.
Why it matters: This is why you can initialize or update any Python dictionary using another dictionary, a list of 2-tuples, a generator yielding pairs, or a custom class that simply implements .keys().
2. The Signature Difference of popitem()
We noted that standard dictionaries are now ordered, reducing the need for collections.OrderedDict. However, their popitem() methods are structurally different:
Standard dict.popitem() accepts no arguments and always removes and returns the last inserted key-value pair (LIFO order).
OrderedDict.popitem(last=True) accepts an optional last keyword argument. If you call .popitem(last=False), it removes and returns the first item inserted (FIFO order). This optimization is why OrderedDict remains the ideal building block for implementing LRU/FIFO caches.
3. Persistent Mappings via shelve.Shelf
The book briefly highlights shelve.Shelf under the "Variations of dict" section.
What it is: It is a persistent mapping stored on disk, mapping string keys to serialized Python objects.
The Mechanism: It subclasses abc.MutableMapping and wraps a DBM database, using the standard library's pickle module under the hood. It is a context manager, meaning you can open it within a with block to ensure it cleanly flushes and closes:
import shelve
with shelve.open('my_db') as shelf:
    shelf['key'] = {'complex': 'object'}
4. Inconsistent __missing__ in the Standard Library
While we thoroughly analyzed how subclassing dict versus UserDict changes lookup behaviors, the book warns that subclassing other standard library mappings can lead to highly inconsistent behavior. Depending on how the C-level or Python-level base class implements methods like get(), __contains__, setdefault(), or update(), they may or may not route through your custom __missing__ fallback. If you build complex subclasses of standard collections, you must explicitly audit or override these methods to enforce consistency